# ECA-0.3 — Parallel Matrix Dynamics (T4 Colab)

**One-click training notebook.** Open this notebook in Google Colab, select a T4 GPU, and run the cells from top to bottom.

This experiment uses raw UTF-8 bytes, batched matrix projections, causal depthwise convolution, gated feature blocks, and an associative affine scan for persistent state. It does **not** use Transformer blocks, self-attention, positional embeddings, BPE, WordPiece, or SentencePiece.

ECA-0.3 is experimental. It has not yet been shown to outperform ECA-0.2. This notebook measures validation bits-per-byte, training throughput, and peak VRAM. The recurrent state carries across chunks; the sequence length is a training segment size, not a fixed inference context window.

In [ ]:
import os, sys, torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
REPO = '/content/eca'
BRANCH = 'eca-0.3-parallel-matrix-dynamics'
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH https://github.com/swasthik-s/eca.git $REPO
%cd /content/eca
!git fetch origin
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH
!pip -q install -e '.[train,dev]'
print('Checked out:', BRANCH)

## 1. Run architecture tests

Do not start training if these tests fail. They check the parallel scan against a sequential recurrence, chunk continuity, causal behavior, gradients, and absence of attention modules.

In [ ]:
!pytest -q tests/test_parallel_matrix.py

## 2. Mount Google Drive for persistent checkpoints

Colab local storage disappears when the runtime resets. The trained checkpoint will be saved in your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT = '/content/drive/MyDrive/ECA/checkpoints/eca_parallel_matrix_0_3.pt'
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
print('Checkpoint:', CHECKPOINT)

## 3. Configure the experiment

The defaults are close to the parameter count of the previous ECA-0.2 T4 run, so the first comparison is less confounded by model size. Start with 300 steps. Increase the data budget only after confirming that the pipeline runs and the GPU has sufficient headroom.

In [ ]:
DATASET_ID = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
DOCUMENTS = 2000
MAX_BYTES = 2_000_000
STEPS = 300
SEQUENCE_LENGTH = 256  # TBPTT segment size for training memory, not an inference context limit
BATCH_SIZE = 8
HIDDEN_SIZE = 144
EMBEDDING_SIZE = 32
DEPTH = 1
KERNEL_SIZE = 5
SEED = 17
print('Training bytes:', MAX_BYTES, '| steps:', STEPS, '| batch:', BATCH_SIZE)

## 4. Train ECA-0.3

The dataset is streamed from Hugging Face. The full FineWeb-Edu corpus is not downloaded. Training prints validation BPB, bytes per second, write intensity, and peak allocated VRAM.

In [ ]:
!python -m eca.training.train_parallel \
  --dataset hf \
  --dataset-id $DATASET_ID \
  --dataset-config $DATASET_CONFIG \
  --documents $DOCUMENTS \
  --max-bytes $MAX_BYTES \
  --steps $STEPS \
  --sequence-length $SEQUENCE_LENGTH \
  --batch-size $BATCH_SIZE \
  --hidden-size $HIDDEN_SIZE \
  --embedding-size $EMBEDDING_SIZE \
  --depth $DEPTH \
  --kernel-size $KERNEL_SIZE \
  --seed $SEED \
  --output "$CHECKPOINT"

## 5. Verify the saved checkpoint

This confirms that the checkpoint exists, can be loaded, and contains the expected ECA-0.3 model weights and configuration. This is not a reasoning or intelligence evaluation.

In [ ]:
import os, torch
assert os.path.isfile(CHECKPOINT), f'Checkpoint not found: {CHECKPOINT}'
saved = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
print('Architecture:', saved['architecture'])
print('Training step:', saved['step'])
print('Configuration:', saved['config'])
print('Parameter tensors:', len(saved['model']))
print('Checkpoint size MB:', round(os.path.getsize(CHECKPOINT) / (1024**2), 2))

## 6. Record the experiment

Save the full cell output. The important comparison is ECA-0.2 versus ECA-0.3 under the same dataset, seed, training byte budget, batch size, sequence length, and GPU runtime.

**Interpretation:** lower validation BPB is better for byte prediction; higher bytes/second indicates more throughput; lower peak VRAM leaves more room for scaling. GPU utilization should be checked separately with `!nvidia-smi -l 1`. A faster model with worse BPB is not automatically an improvement.

**Known limitation:** the current write-intensity penalty discourages memory writes but does not yet skip the associated GPU computation. Persistent state is finite, and no architecture has infinite memory.